<a href="https://colab.research.google.com/github/Springboard429/B12---PlantDocBot-AI-Plant-Disease-Diagnosis-via-Chat-and-Image-Upload/blob/Sneha_John/plantdoc5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Required Libraries

In [8]:
!pip install transformers datasets scikit-learn pandas pyarrow evaluate joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


Import Necessary Libraries

In [9]:
import pandas as pd
import numpy as np
import torch
import ast
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

import evaluate

Load Dataset From Parquet File

In [10]:
df = pd.read_parquet("/content/drive/MyDrive/Infosys/train-00000-of-00001.parquet")

print(df.columns)
df.head()

Index(['image', 'caption', 'captions'], dtype='object')


,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


Extract Text Descriptions and Labels

In [11]:
df = df.explode("captions", ignore_index=True)

df = df.rename(columns={
    "captions": "text",
    "caption": "label_name"
})

df = df[["text","label_name"]]

print("Total samples:", len(df))
print("Unique diseases:", df["label_name"].nunique())

Total samples: 82552
Unique diseases: 15


Encode Disease Labels

In [12]:
encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["label_name"])

num_classes = len(encoder.classes_)

print("Number of classes:", num_classes)
print(encoder.classes_)

Number of classes: 15
['Pepper bell Bacterial spot' 'Pepper bell healthy' 'Potato Early blight'
 'Potato Late blight' 'Potato healthy' 'Tomato Bacterial spot'
 'Tomato Early blight' 'Tomato Late blight' 'Tomato Leaf Mold'
 'Tomato Septoria leaf spot' 'Tomato Spider mites Two spotted spider mite'
 'Tomato Target Spot' 'Tomato YellowLeaf Curl Virus' 'Tomato healthy'
 'Tomato mosaic virus']


Train-Test Split

In [13]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 66041
Test size: 16511


Tokenize Text Using DistilBERT Tokenizer

In [14]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","label"]
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

Load Pretrained DistilBERT Model

In [15]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_classes
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Configure Training Arguments

In [16]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Initialize Trainer

In [17]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000211,0.000087,1.000000
2,0.000023,0.000008,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000211,0.000087,1.000000
2,0.000023,0.000008,1.000000
3,0.000003,0.000001,1.000000
4,0.000001,0.000000,1.000000
5,0.000000,0.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=20640, training_loss=0.012532869324363215, metrics={'train_runtime': 3969.0425, 'train_samples_per_second': 83.195, 'train_steps_per_second': 5.2, 'total_flos': 1.09378845569088e+16, 'train_loss': 0.012532869324363215, 'epoch': 5.0})

Save the Trained Model

In [18]:
trainer.save_model("/content/drive/MyDrive/Infosys/distilbert_plant_disease_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Infosys/distilbert_plant_disease_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Infosys/distilbert_plant_disease_model/tokenizer_config.json',
 '/content/drive/MyDrive/Infosys/distilbert_plant_disease_model/tokenizer.json')

Save LabelEncoder

In [25]:
joblib.dump(encoder, "/content/drive/MyDrive/Infosys/label_encoder.pkl")

['/content/drive/MyDrive/Infosys/label_encoder.pkl']

Create text_inference.py

In [39]:
%%writefile text_inference.py

import torch
import joblib
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# Load saved model
model_path = "/content/drive/MyDrive/Infosys/distilbert_plant_disease_model"

tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)
model = DistilBertForSequenceClassification.from_pretrained(model_path)

# Load label encoder
encoder = joblib.load("/content/drive/MyDrive/Infosys/label_encoder.pkl")

model.eval()

def predict_disease(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    disease = encoder.inverse_transform([prediction])[0]

    return disease


if __name__ == "__main__":

    text = input("Enter plant disease description: ")

    result = predict_disease(text)

    print("Predicted Disease:", result)

Overwriting text_inference.py


In [40]:
!cp text_inference.py /content/drive/MyDrive/Infosys/
!python text_inference.py

Loading weights: 100% 104/104 [00:00<00:00, 1172.27it/s, Materializing param=pre_classifier.weight]
Enter plant disease description: A tomato plant leaf curling and yellowing, typical
Predicted Disease: Tomato YellowLeaf Curl Virus


In [ ]:
from google.colab import files
files.download("text_inference.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
files.download("/content/drive/MyDrive/Infosys/label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

# Define paths
model_dir_path = "/content/drive/MyDrive/Infosys/distilbert_plant_disease_model"
output_zip_path = "/content/drive/MyDrive/Infosys/distilbert_plant_disease_model.zip"

# Navigate to the parent directory to zip the folder correctly
# os.chdir("/content/drive/MyDrive/Infosys/")

# Compress the directory using a shell command
# The -r flag is for recursive inclusion of directory contents
# The -j flag is for junking directory names (flat structure) if not desired, but here we want the folder structure
# We will use -r to include the directory and its contents, and specify the full path of the model directory.
!zip -r "{output_zip_path}" "{model_dir_path}"

print(f"Successfully compressed '{model_dir_path}' to '{output_zip_path}'")

  adding: content/drive/MyDrive/Infosys/distilbert_plant_disease_model/ (stored 0%)
  adding: content/drive/MyDrive/Infosys/distilbert_plant_disease_model/config.json (deflated 61%)
  adding: content/drive/MyDrive/Infosys/distilbert_plant_disease_model/model.safetensors (deflated 8%)
  adding: content/drive/MyDrive/Infosys/distilbert_plant_disease_model/training_args.bin (deflated 53%)
  adding: content/drive/MyDrive/Infosys/distilbert_plant_disease_model/tokenizer_config.json (deflated 42%)
  adding: content/drive/MyDrive/Infosys/distilbert_plant_disease_model/tokenizer.json (deflated 71%)
Successfully compressed '/content/drive/MyDrive/Infosys/distilbert_plant_disease_model' to '/content/drive/MyDrive/Infosys/distilbert_plant_disease_model.zip'


In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/Infosys/distilbert_plant_disease_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Predict Disease from Text Description

In [57]:
def predict_disease(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = torch.argmax(outputs.logits, dim=1).item()

    disease = encoder.inverse_transform([predicted_class])[0]

    return disease


sample = "yellow spots on tomato leaves "

print("Predicted Disease:", predict_disease(sample))

Predicted Disease: Tomato Bacterial spot
